# RAG 2
Multiple data source (PDF: Apple_form_10K.pdf, Excel: Sample_Financial_Data.xlsx)

In [76]:
from IPython.display import display, Markdown
from pathlib import Path
import pandas as pd 

import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [13]:
apple_doc_path = list(Path.cwd().rglob("Apple_form_10k.pdf"))
financial_data_path = list(Path.cwd().rglob("Sample_Financial_Data.xlsx"))

## Parsing and Chunking of PDF

> **NOTE:** As we want to add metadata to our chunks, so will be wrapping our chunks LangChain's **Document** class and add metadata like the file name and file type.

In [14]:
md_text = pymupdf4llm.to_markdown(str(apple_doc_path[0]))

In [69]:
# Chunking
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
pdf_chunks = text_splitter.split_text(md_text)

In [70]:
pdf_chunks[0]

Document(metadata={'Header 1': '**UNITED STATES SECURITIES AND EXCHANGE COMMISSION**'}, page_content='**Washington, D.C. 20549**')

In [71]:
# Since the splitter alreadt wraps the chunks in Document class, we can add more metadata to each chunk
for i in pdf_chunks:
    i.metadata["source"] = "apple_10k.md"
    i.metadata["doc_type"] = "PDF-Markdown"

In [72]:
pdf_chunks[0].metadata

{'Header 1': '**UNITED STATES SECURITIES AND EXCHANGE COMMISSION**',
 'source': 'apple_10k.md',
 'doc_type': 'PDF-Markdown'}

## Parsing and Chunking of Excel 

In [36]:
financial_data = pd.read_excel(str(financial_data_path[0]))

In [38]:
financial_data.head()

,Segment,Country,Product,Discount Band,Units Sold,Manufacturing Price,Sale Price,Gross Sales,Discounts,Sales,COGS,Profit,Date,Month Number,Month Name,Year
0,Government,Canada,Carretera,NaN,1618.5,3,20,32370.0,0.0,32370.0,16185.0,16185.0,2014-01-01,1,January,2014
1,Government,Germany,Carretera,NaN,1321.0,3,20,26420.0,0.0,26420.0,13210.0,13210.0,2014-01-01,1,January,2014
2,Midmarket,France,Carretera,NaN,2178.0,3,15,32670.0,0.0,32670.0,21780.0,10890.0,2014-06-01,6,June,2014
3,Midmarket,Germany,Carretera,NaN,888.0,3,15,13320.0,0.0,13320.0,8880.0,4440.0,2014-06-01,6,June,2014
4,Midmarket,Mexico,Carretera,NaN,2470.0,3,15,37050.0,0.0,37050.0,24700.0,12350.0,2014-06-01,6,June,2014


In [53]:
# Row-as-a-string method
excel_chunks = []

for index, row in financial_data.iterrows():
	row_text = (
			f"In {row['Month Name']} {row['Year']}, the {row['Segment']} segment in {row['Country']} "
			f"purchased {row['Units Sold']:,} units of the product '{row['Product']}'. "
			f"The manufacturing price was ${row['Manufacturing Price']} and the sale price was ${row['Sale Price']}. "
			f"This transaction generated ${row['Gross Sales']:,} in gross sales with a discount band of ${row['Discount Band']} "
			f"amounting to ${row['Discounts']:,} in total discounts. Net sales were ${row[' Sales']}. "
			f"The Cost of Goods Sold (COGS) was ${row['COGS']:,}, resulting in a net profit of ${row['Profit']:,}."
		)
	
	doc = Document(
			page_content=row_text,
			metadata={
					"source": "financial_data.xlsx",
					"row_index": index,
					"doc_type": "Excel",
					"country": row['Country'],
					"product": row['Product']
			}
		)

	excel_chunks.append(doc)

In [73]:
excel_chunks[0]

Document(metadata={'source': 'financial_data.xlsx', 'row_index': 0, 'doc_type': 'Excel', 'country': 'Canada', 'product': 'Carretera'}, page_content="In January 2014, the Government segment in Canada purchased 1,618.5 units of the product 'Carretera'. The manufacturing price was $3 and the sale price was $20. This transaction generated $32,370.0 in gross sales with a discount band of $nan amounting to $0.0 in total discounts. Net sales were $32370.0. The Cost of Goods Sold (COGS) was $16,185.0, resulting in a net profit of $16,185.0.")

## Embedding in V DB

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [99]:
pdf_chunk_ids = [f"pdf_chunk_{i}" for i in range(len(pdf_chunks))]
excel_chunk_ids = [f"excel_chunk_{i}" for i in range(len(excel_chunks))]

In [111]:
db_name = Path.cwd() / "vdb" 

if db_name.exists():
    Chroma(persist_directory=str(db_name), embedding_function=embeddings).delete_collection()

vector_store = Chroma(
    collection_name="embeddings",
    embedding_function=embeddings,
    persist_directory=str(db_name)
)

vector_store.add_documents(documents=pdf_chunks, ids=pdf_chunk_ids)
vector_store.add_documents(documents=excel_chunks, ids=excel_chunk_ids)

print(f"Vectorstore created with {vector_store._collection.count()} documents")

Vectorstore created with 917 documents


## Retrieval

In [135]:
result = vector_store.similarity_search("Why developers tend to devlop more non-apple apps?", k=5)

In [137]:
result

[Document(id='pdf_chunk_37', metadata={'Header 2': '_**The Company’s future performance depends in part on support from third-party software developers.**_', 'Header 1': 'For the fiscal year ended September 27, 2025', 'doc_type': 'PDF-Markdown', 'source': 'apple_10k.md'}, page_content='The Company believes decisions by customers to purchase its hardware products depend in part on the availability of third-party software applications and services. Third-party developers may discontinue the development and maintenance of software applications and services for the Company’s products. If third-party software applications and services cease to be developed and maintained for the Company’s products, customers may choose not to buy the Company’s products, adversely impacting the Company’s business, results of operations, financial condition and stock price.  \nThe Company believes that third-party developer support depends on the perceived benefits of creating software and services for the Co

In [136]:
[i.page_content for i in result]

['The Company believes decisions by customers to purchase its hardware products depend in part on the availability of third-party software applications and services. Third-party developers may discontinue the development and maintenance of software applications and services for the Company’s products. If third-party software applications and services cease to be developed and maintained for the Company’s products, customers may choose not to buy the Company’s products, adversely impacting the Company’s business, results of operations, financial condition and stock price.  \nThe Company believes that third-party developer support depends on the perceived benefits of creating software and services for the Company’s products compared to competitors’ platforms, such as Android for smartphones and tablets, Windows for personal computers and tablets, and PlayStation, Nintendo and Xbox for gaming platforms. This analysis may be based on factors such as the market position of the Company and i